# ARC Dataset EDA — CSE 151B Project

Produces the statistics and figures referenced in `IMPLEMENTATION_PLAN.md` section 2.3:
split sizes, choice-count distribution, answer-position distribution, and BERT token
length distribution (to justify `max_seq_length = 128`). Figures are saved to
`../results/figures/` for the paper.

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from data import load_arc

FIG_DIR = Path.cwd().parent / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

splits = ["train", "validation", "test"]
data = {(sub, sp): load_arc(sub, sp) for sub in ["easy", "challenge"] for sp in splits}

stats = pd.DataFrame(
    {sub: {sp: len(data[(sub, sp)]) for sp in splits} for sub in ["easy", "challenge"]}
).T
stats["total"] = stats.sum(axis=1)
stats.loc["combined"] = stats.sum()
stats  # expect: easy 2251/570/2376, challenge 1119/299/1172, total 7787

In [ ]:
# Choice-count distribution — motivates the variable-choice collator (plan 2.2)
rows = []
for (sub, sp), ds in data.items():
    for ex in ds:
        rows.append({"subset": sub, "split": sp, "n_choices": len(ex["choices"]),
                     "answer_pos": ex["label"]})
df = pd.DataFrame(rows)
print(df.groupby(["subset", "n_choices"]).size().unstack(fill_value=0))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
df.groupby(["subset", "n_choices"]).size().unstack(fill_value=0).T.plot.bar(ax=axes[0], rot=0)
axes[0].set_xlabel("number of choices"); axes[0].set_ylabel("questions")
axes[0].set_title("Choice counts (all splits)")
df[df.n_choices == 4].groupby(["subset", "answer_pos"]).size().unstack(fill_value=0).T.plot.bar(ax=axes[1], rot=0)
axes[1].set_xlabel("correct answer position (4-choice questions)"); axes[1].set_ylabel("questions")
axes[1].set_title("Answer position distribution")
fig.tight_layout()
fig.savefig(FIG_DIR / "choices_and_positions.png", dpi=200)
plt.show()

In [ ]:
# BERT token lengths of question+choice pairs -> justifies max_seq_length and
# measures the truncation rate (plan 2.3 / 4.2)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LENGTH = 128

lengths = []
for (sub, sp), ds in data.items():
    for ex in ds:
        for choice in ex["choices"]:
            enc = tokenizer(ex["question"], choice)
            lengths.append(len(enc["input_ids"]))
lengths = pd.Series(lengths)

trunc_rate = (lengths > MAX_LENGTH).mean()
print(f"pairs: {len(lengths)}, mean {lengths.mean():.1f} tokens, "
      f"p95 {lengths.quantile(0.95):.0f}, p99 {lengths.quantile(0.99):.0f}, max {lengths.max()}")
print(f"truncation rate at {MAX_LENGTH}: {trunc_rate:.2%}  (plan: raise to 192 only if > 2%)")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(lengths, bins=60)
ax.axvline(MAX_LENGTH, color="k", linestyle="--", label=f"max_seq_length = {MAX_LENGTH}")
ax.set_xlabel("BERT tokens per (question, choice) pair"); ax.set_ylabel("count")
ax.set_title(f"Token lengths (truncation rate {trunc_rate:.2%})")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "token_lengths.png", dpi=200)
plt.show()

**For the paper:** the split-size table (cell 2), both figures, and the truncation-rate
number feed the Dataset section. If the truncation rate at 128 exceeds ~2%, retrain with
`--max_length 192` and note the change.